# 🧠 Conversational Agent with Memory — Step-by-Step Guide

**What is Agent Memory?**  
Standard LLMs and basic agents are completely **stateless**. Each user interaction (`invoke`) occurs in total isolation, with zero memory of what was said in prior turns.

If a user asks:
1. *Turn 1:* "What is the price of wireless headphones?"  
2. *Turn 2:* "What are the reviews on this product?"

A stateless agent will fail on Turn 2 because it cannot resolve what **"this product"** refers to.

**LangGraph State Checkpointing** solves this by maintaining a persistent conversational history:
- **`InMemorySaver`**: A checkpointer that stores conversation state frames in memory.
- **`thread_id`**: A unique session identifier (e.g., `user-alice-session-1`). When calls share the same `thread_id`, the agent automatically retrieves prior conversational context, resolves coreferences/pronouns, and remembers past decisions.

---

### Conversational Memory Architecture

```
                                 ┌──────────────────────────────────────┐
                                 │   State Checkpointer (InMemorySaver) │
                                 │   Thread ID: user-alice-session-1    │
                                 └───────────────▲──────────────────────┘
                                                 │ Reads / Writes State
                                                 ▼
┌──────────────────────────────┐          ┌──────────────┐          ┌───────────────────────────┐
│ Turn 1 User Prompt           │ ───────► │ ReAct Agent  │ ───────► │ Agent Response:           │
│ "What is the price of        │          │   + Tools    │          │ "Wireless headphones      │
│  wireless headphones?"       │          └──────────────┘          │  cost $79.99"             │
└──────────────────────────────┘                                    └───────────────────────────┘
                                                 │
                             Session checkpoint persisted to thread_id
                                                 │
                                                 ▼
┌──────────────────────────────┐          ┌──────────────┐          ┌───────────────────────────┐
│ Turn 2 Follow-Up Prompt      │ ───────► │ Memory-Aware │ ───────► │ Agent Response:           │
│ "What are the reviews on     │          │    Agent     │          │ "The wireless headphones  │
│  this product?"              │          │              │          │  have a 4.6/5 rating from │
│  (Pronoun: "this product")   │          └──────┬───────┘          │  1,262 reviews."          │
└──────────────────────────────┘                 │                  └───────────────────────────┘
                                                 │
                                                 ▼ [Coreference Resolved: "this product" = headphones]
                                    Calls get_review("wireless headphones")
```

**Key Concepts Covered:**
1. **Stateless vs. Stateful Agents**: Demonstrating the contrast between one-shot execution and continuous dialogue.
2. **LangGraph Checkpointing**: Attaching an `InMemorySaver` to persist graph state between turns.
3. **Session Partitioning via `thread_id`**: Isolating conversations so distinct users or sessions maintain independent conversation histories.
4. **Coreference Resolution**: Enabling the agent to understand pronouns like *"it"*, *"this product"*, or *"that one"* from previous dialogue.


---
## Step 1 — Setup & Environment Configuration

We load environment variables and import agent and tool dependencies.


In [8]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent


load_dotenv()


True

---
## Step 2 — Product Knowledge Base & Tool Definition

We configure the product catalog and the `get_product` tool.


In [9]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name : str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found, Available: { ', '.join(PRODUCTS)}"
    return str(p)
    

---
## Step 3 — Baseline Stateless Agent (No Memory)

First, we create a baseline agent without a checkpointer. This agent evaluates each prompt independently without maintaining session history.


In [10]:
    
    llm = ChatOpenAI(
        model="gpt-6-luna",
        # temperature=0,
    )
    
    agent = create_agent(
        llm,
        tools = [get_product], 
        system_prompt="You are a helpful product assistant for an online test store",
    )

---
## Step 4 — Invocation Helper & Stateless Queries

We run test queries against the stateless agent.


In [11]:
def get_text(content):
    if isinstance(content, list):
        return "\n".join(item["text"] for item in content if isinstance(item, dict) and item.get("type") == "text")
    return content

def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(get_text(result["messages"][-1].content))


In [12]:
ask("what is the price of wireless headphones")

The wireless headphones cost $79.99.


In [13]:
ask("tell me about mechanical keyboard")

The mechanical keyboard is a tenkeyless model with Cherry MX Brown switches and per-key RGB lighting. It’s rated 4.8/5 and costs $129.00.


---
## Step 5 — Review Tool Definition

We define `get_review` to support customer review queries.


In [14]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}


@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)

---
## Step 6 — Memory-Enabled Agent with `InMemorySaver` & Session Checkpointing

Here is the key architectural transition:
1. We pass `checkpointer=InMemorySaver()` into `create_agent`.
2. When invoking `agent2`, we supply a session configuration:  
   `config = {"configurable": {"thread_id": "user-alice-session-1"}}`
3. All messages under this `thread_id` are saved and re-injected automatically into subsequent turns.


In [21]:
# from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.memory import InMemorySaver


llm = ChatOpenAI(
    model="gpt-6-luna",
    # temperature=0,
)
    

agent2 = create_agent(
    llm,
    tools = [get_product, get_review],
    system_prompt = "You are a helpful product assistant for an online tech store.",
    checkpointer = InMemorySaver(),
)

def ask2(question: str):
    config = {"configurable": {"thread_id" : "user-alice-session-1"}}
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]}, config)
    print(get_text(result["messages"][-1].content))


---
## Step 7 — Multi-Turn Conversational Verification & Coreference Resolution

Observe the multi-turn dialogue below:
- **Turn 1**: The user asks for the price of *wireless headphones*.
- **Turn 2**: The user asks: *"what are the reviews on this product"*.
  Notice the user **never mentions** the product name! Because of the checkpointer and `thread_id`, the agent recalls the previous turn, resolves *"this product"* to *wireless headphones*, and calls `get_review("wireless headphones")`!


In [25]:
ask2("what is the price of wireless headphones")

Wireless headphones cost **$79.99**.


In [26]:
ask2("what are the reviews on this product")

The wireless headphones have a **4.6/5 rating from 1,262 reviews**.
